# Agentic RAG with Follow-Up Retrieval

## A RAG System That Decides When to Retrieve More Context

**Project 23** - Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Agentic retrieval with multi-round decision loop |
| Agent pattern | State machine with Retrieve-Evaluate-Reformulate nodes |
| Retrieval | Dense embedding search (sentence-transformers) |
| Corpus | 30-document TechStart knowledge base |
| Evaluation | Single-pass vs agentic retrieval comparison |


## Project Overview

Standard RAG retrieves once and answers. But some questions need:

- A **second retrieval** with a refined query
- Context from a **different topic area**
- **Multiple pieces** of evidence stitched together

**Agentic RAG** adds a decision loop: after the first retrieval,
the system evaluates whether it has enough context. If not, it
reformulates the query and retrieves again.

```
User Question
     |
     v
  RETRIEVE (round 1)
     |
     v
  EVALUATE: Sufficient? ---No---> REFORMULATE query
     |                                   |
     | Yes                               v
     v                            RETRIEVE (round 2)
  GENERATE answer                        |
     ^                                   v
     |                            EVALUATE: Sufficient?
     +------- Yes <------ Yes -----------+
                                  |
                                  No (max rounds)
                                  |
                                  v
                           GENERATE (partial)
```

This notebook implements the full agent loop **without** external
LLM dependencies. We use heuristic sufficiency scoring to demonstrate
the decision logic transparently.


## Learning Goals

1. Understand **why** single-pass retrieval fails for complex questions
2. Implement a state-machine agent with Retrieve, Evaluate, and Reformulate nodes
3. Design **sufficiency scoring** - how the agent decides "enough context"
4. Compare single-pass vs agentic multi-pass retrieval
5. Understand the LangGraph paradigm through a from-scratch implementation


## Problem Statement

Given a knowledge base about a company's product suite, users ask:

- **Cross-topic questions**: "Compare pricing and performance" - needs docs from two topic areas
- **Follow-up-dependent questions**: "What security certifications exist and how do they affect pricing?"
- **Multi-evidence questions**: "List all databases supported with their performance characteristics"

A single retrieval round often misses one topic entirely.
The agent must detect this gap and retrieve again.


## Why Agentic RAG Matters

| Scenario | Single-pass RAG | Agentic RAG |
|----------|----------------|-------------|
| Simple factual Q | Works fine | Works fine (1 round) |
| Cross-topic Q | Misses one topic | Detects gap, retrieves more |
| Ambiguous Q | Answers wrong aspect | Reformulates for precision |
| Evidence-heavy Q | Partial answer | Gathers sufficient evidence |

In production, agentic RAG typically uses an LLM to decide sufficiency.
Here we use heuristic scoring to make the decision logic **fully transparent**.


## Understanding Agent Graphs (LangGraph Concepts)

LangGraph models agents as **stateful directed graphs**:

- **State**: A typed dictionary that flows through the graph
- **Nodes**: Functions that read state, do work, and return state updates
- **Edges**: Connections between nodes (can be conditional)
- **Conditional edges**: Route to different nodes based on state

Our agent graph has **3 nodes** and **2 conditional edges**:

```
Nodes:
  [RETRIEVE]    - embed query, find top-K docs
  [EVALUATE]    - score sufficiency of retrieved context
  [REFORMULATE] - generate a new query targeting missing info

Edges:
  START --> RETRIEVE
  RETRIEVE --> EVALUATE
  EVALUATE --sufficient--> ANSWER (end)
  EVALUATE --insufficient--> REFORMULATE
  REFORMULATE --> RETRIEVE
```

We implement this **without** the LangGraph library to show exactly
what happens inside. The pattern maps directly to LangGraph if you
want to use it in production.


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random, time, json
from typing import List, Dict, Optional
from dataclasses import dataclass, field

import numpy as np
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K = 3               # docs per retrieval round
MAX_ROUNDS = 3      # max retrieval rounds before forcing an answer
SUFFICIENCY_THRESHOLD = 0.65  # minimum coverage score to stop retrieving

print(f"Config: model={EMBEDDING_MODEL}, K={K}, max_rounds={MAX_ROUNDS}, "
      f"sufficiency_threshold={SUFFICIENCY_THRESHOLD}")


## Knowledge Base

A 30-document corpus about **TechStart**, a fictional SaaS company.
Documents span 6 topic areas so cross-topic questions require
multiple retrieval rounds.


In [ ]:
corpus = [
    # --- PRICING (P01-P05) ---
    {"id": "P01", "text": "TechStart pricing has three tiers: Free (5 users, 1 GB), Pro at $29/month (50 users, 100 GB), and Enterprise at custom pricing with unlimited users and storage.", "topic": "pricing"},
    {"id": "P02", "text": "TechStart Pro plan includes priority email support, SSO integration, and API access. Annual billing saves 20% compared to monthly.", "topic": "pricing"},
    {"id": "P03", "text": "TechStart Enterprise includes a dedicated account manager, custom SLA (up to 99.99% uptime), on-premise deployment option, and volume discounts for 500+ seats.", "topic": "pricing"},
    {"id": "P04", "text": "TechStart offers a 14-day free trial of the Pro plan. No credit card required. Trial accounts convert to Free tier automatically.", "topic": "pricing"},
    {"id": "P05", "text": "TechStart educational pricing gives 50% discount to verified academic institutions. Non-profit organizations receive 30% off Enterprise pricing.", "topic": "pricing"},

    # --- DATABASE (D01-D05) ---
    {"id": "D01", "text": "TechStart DataPipeline supports PostgreSQL 15, MySQL 8, MongoDB 7, and Redis 7 as backend stores. PostgreSQL is the default and recommended for most workloads.", "topic": "database"},
    {"id": "D02", "text": "PostgreSQL performance on TechStart: 12,000 read queries/sec and 3,000 write queries/sec on the standard instance. Connection pooling via PgBouncer handles up to 10,000 concurrent connections.", "topic": "database"},
    {"id": "D03", "text": "MongoDB on TechStart supports sharding for horizontal scaling. Recommended for document-heavy workloads exceeding 1 TB. Available on Pro and Enterprise only.", "topic": "database"},
    {"id": "D04", "text": "TechStart Redis integration provides caching with sub-millisecond latency. L1 cache TTL is 5 minutes, L2 cache TTL is 1 hour. Cache hit ratio averages 94% in production.", "topic": "database"},
    {"id": "D05", "text": "Database migration support includes zero-downtime schema migrations, automatic backups every 6 hours, and point-in-time recovery up to 30 days on Enterprise.", "topic": "database"},

    # --- SECURITY (S01-S05) ---
    {"id": "S01", "text": "TechStart uses AES-256 encryption at rest and TLS 1.3 in transit. All encryption keys are managed through AWS KMS with automatic rotation every 90 days.", "topic": "security"},
    {"id": "S02", "text": "TechStart holds SOC 2 Type II, ISO 27001, and GDPR certifications. HIPAA BAA is available on Enterprise plans. Annual penetration testing is conducted by NCC Group.", "topic": "security"},
    {"id": "S03", "text": "TechStart supports SAML 2.0 SSO, OpenID Connect, and LDAP integration. Multi-factor authentication is available on all plans using TOTP or WebAuthn.", "topic": "security"},
    {"id": "S04", "text": "TechStart RBAC has four roles: Admin, Editor, Viewer, and Auditor. Custom roles with fine-grained permissions are available on Enterprise.", "topic": "security"},
    {"id": "S05", "text": "TechStart audit logging captures all user actions, API calls, and configuration changes. Logs are retained 90 days on Pro and 1 year on Enterprise. SIEM export is supported.", "topic": "security"},

    # --- PERFORMANCE (R01-R05) ---
    {"id": "R01", "text": "TechStart API average response time is 45ms at P50 and 120ms at P99. The global CDN reduces latency to under 30ms for cached content in all major regions.", "topic": "performance"},
    {"id": "R02", "text": "TechStart auto-scaling handles traffic spikes up to 10x baseline within 60 seconds. Horizontal pod autoscaling uses CPU and memory metrics with custom thresholds.", "topic": "performance"},
    {"id": "R03", "text": "TechStart SLA guarantees 99.9% uptime on Pro and 99.99% on Enterprise. Planned maintenance windows are Sundays 02:00-04:00 UTC with zero-downtime deployments.", "topic": "performance"},
    {"id": "R04", "text": "TechStart file sync throughput: 500 MB/s for sequential reads, 200 MB/s for writes. Chunked transfer with resumable uploads handles files up to 50 GB.", "topic": "performance"},
    {"id": "R05", "text": "TechStart monitoring uses Prometheus and Grafana. Key dashboards track request latency, error rates, queue depth, and database connection pool utilization.", "topic": "performance"},

    # --- DEPLOYMENT (T01-T05) ---
    {"id": "T01", "text": "TechStart deploys via Docker and Kubernetes. Helm charts are provided for standard configurations. The recommended production setup uses 3 replicas with HPA.", "topic": "deployment"},
    {"id": "T02", "text": "TechStart CI/CD integrates with GitHub Actions, GitLab CI, and Jenkins. Blue-green deployments are supported. Canary releases roll out to 5% of traffic first.", "topic": "deployment"},
    {"id": "T03", "text": "TechStart on-premise deployment is available on Enterprise. Requirements: 8 CPU cores, 32 GB RAM, 500 GB SSD minimum. Air-gapped environments are supported.", "topic": "deployment"},
    {"id": "T04", "text": "TechStart supports multi-region deployment across US, EU, and APAC. Data residency controls ensure compliance with GDPR and local data sovereignty laws.", "topic": "deployment"},
    {"id": "T05", "text": "TechStart disaster recovery uses cross-region replication with RPO of 1 hour and RTO of 4 hours. Enterprise gets RPO of 5 minutes with automatic failover.", "topic": "deployment"},

    # --- API & SDK (A01-A05) ---
    {"id": "A01", "text": "TechStart REST API uses OAuth 2.0 with JWT tokens. Rate limits: 1000 req/min on Free, 5000 on Pro, unlimited on Enterprise. GraphQL API is in beta.", "topic": "api"},
    {"id": "A02", "text": "TechStart Python SDK supports async operations via asyncio. Available on PyPI. Covers all REST endpoints with type hints and auto-retry with exponential backoff.", "topic": "api"},
    {"id": "A03", "text": "TechStart JavaScript SDK is available on npm. Supports Node.js 18+ and browser environments. Includes TypeScript definitions and tree-shaking support.", "topic": "api"},
    {"id": "A04", "text": "TechStart webhooks deliver events via HTTP POST: sync_complete, sync_failed, user_added, file_conflict, quota_exceeded. Retry uses exponential backoff up to 5 attempts.", "topic": "api"},
    {"id": "A05", "text": "TechStart API versioning uses URL path versioning (v1, v2). Breaking changes are announced 6 months in advance. v1 sunset date is December 2026.", "topic": "api"},
]

# Build topic index for analysis
topic_docs = {}
for doc in corpus:
    topic_docs.setdefault(doc["topic"], []).append(doc["id"])

print(f"Knowledge base: {len(corpus)} documents across {len(topic_docs)} topics")
for topic, ids in topic_docs.items():
    print(f"  {topic:>12s}: {ids}")


## Dense Retriever

Same retrieval engine as projects 21-22: sentence-transformers
with cosine similarity.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

doc_texts = [doc["text"] for doc in corpus]
doc_embeddings = encoder.encode(doc_texts, convert_to_numpy=True, show_progress_bar=False)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)


def retrieve(query: str, k: int = K, exclude_ids: List[str] = None) -> List[Dict]:
    """Retrieve top-k documents. Optionally exclude already-retrieved doc IDs."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (doc_embeddings @ q_emb.T).flatten()
    
    if exclude_ids:
        for i, doc in enumerate(corpus):
            if doc["id"] in exclude_ids:
                scores[i] = -1.0  # suppress already-retrieved
    
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "text": corpus[i]["text"],
             "topic": corpus[i]["topic"], "score": float(scores[i])}
            for i in top_idx]


# Sanity check
test = retrieve("TechStart pricing plans")
print(f"Retriever ready (index: {doc_embeddings.shape})")
print(f"Test query: {[r['id'] for r in test]}")


## Agent State

The agent state is a dictionary that flows through the graph.
In LangGraph, this would be a `TypedDict`. We use a dataclass
for clarity.

Key fields:
- `question` - the original user question
- `search_query` - current query being used for retrieval (may differ from original)
- `retrieved_docs` - all documents retrieved across all rounds
- `topics_covered` - which topic areas we have docs from
- `topics_needed` - which topic areas the question requires
- `round` - current retrieval round
- `sufficiency_score` - how well the context covers the question
- `decision_log` - trace of all agent decisions (for debugging)


In [ ]:
@dataclass
class AgentState:
    """State that flows through the agent graph."""
    question: str
    search_query: str = ""
    retrieved_docs: List[Dict] = field(default_factory=list)
    topics_covered: set = field(default_factory=set)
    topics_needed: set = field(default_factory=set)
    round_num: int = 0
    sufficiency_score: float = 0.0
    is_sufficient: bool = False
    decision_log: List[str] = field(default_factory=list)
    
    @property
    def retrieved_ids(self) -> List[str]:
        return [d["id"] for d in self.retrieved_docs]
    
    @property
    def topics_missing(self) -> set:
        return self.topics_needed - self.topics_covered


print("AgentState defined.")


## Node 1: Topic Detection

Before retrieval, the agent analyzes the question to determine
which topic areas it touches. This lets us evaluate coverage later.

In production, an LLM would do this. Here we use keyword matching
for transparency.


In [ ]:
TOPIC_KEYWORDS = {
    "pricing": ["pricing", "price", "cost", "plan", "tier", "subscription",
                "free", "pro", "enterprise", "billing", "discount", "trial"],
    "database": ["database", "postgresql", "mysql", "mongodb", "redis",
                 "cache", "storage", "data store", "migration", "backup"],
    "security": ["security", "encryption", "ssl", "tls", "soc", "iso",
                 "hipaa", "gdpr", "sso", "saml", "mfa", "authentication",
                 "rbac", "audit", "compliance", "certification", "penetration"],
    "performance": ["performance", "latency", "throughput", "uptime", "sla",
                    "response time", "auto-scaling", "scaling", "speed",
                    "monitoring", "metrics", "p99", "cdn"],
    "deployment": ["deployment", "deploy", "docker", "kubernetes", "helm",
                   "ci/cd", "on-premise", "region", "disaster recovery",
                   "failover", "rpo", "rto"],
    "api": ["api", "sdk", "rest", "graphql", "webhook", "oauth", "jwt",
            "python sdk", "javascript sdk", "rate limit", "endpoint"],
}


def detect_topics(question: str) -> set:
    """Detect which topic areas a question touches."""
    q_lower = question.lower()
    detected = set()
    for topic, keywords in TOPIC_KEYWORDS.items():
        for kw in keywords:
            if kw in q_lower:
                detected.add(topic)
                break
    # If no topics detected, return all (broad question)
    if not detected:
        detected = {"pricing", "database", "security", "performance"}
    return detected


# Test
test_q = "Compare pricing and performance"
detected = detect_topics(test_q)
print(f"Q: {test_q}")
print(f"Topics detected: {detected}")


## Node 2: Retrieve

The RETRIEVE node embeds the current search query and finds top-K docs.
It excludes docs already retrieved in previous rounds so we get
fresh information each time.


In [ ]:
def node_retrieve(state: AgentState) -> AgentState:
    """RETRIEVE node: get top-K docs for current search query."""
    state.round_num += 1
    query = state.search_query or state.question
    
    new_docs = retrieve(query, k=K, exclude_ids=state.retrieved_ids)
    state.retrieved_docs.extend(new_docs)
    
    # Track topics covered
    for doc in new_docs:
        state.topics_covered.add(doc["topic"])
    
    new_ids = [d["id"] for d in new_docs]
    new_topics = {d["topic"] for d in new_docs}
    state.decision_log.append(
        f"RETRIEVE (round {state.round_num}): query='{query}' -> "
        f"docs={new_ids}, topics={new_topics}"
    )
    return state


print("RETRIEVE node defined.")


## Node 3: Evaluate Sufficiency

The EVALUATE node scores how well the retrieved context covers
the question. It checks:

1. **Topic coverage**: Are all needed topic areas represented?
2. **Relevance scores**: Are the retrieved docs actually relevant?
3. **Evidence count**: Do we have enough supporting docs?

The sufficiency score (0 to 1) determines the next action:
- Score >= threshold -> ANSWER (sufficient)
- Score < threshold -> REFORMULATE (need more)


In [ ]:
def node_evaluate(state: AgentState) -> AgentState:
    """EVALUATE node: score sufficiency of retrieved context."""
    # Factor 1: Topic coverage (0-1)
    if state.topics_needed:
        topic_coverage = len(state.topics_covered & state.topics_needed) / len(state.topics_needed)
    else:
        topic_coverage = 1.0
    
    # Factor 2: Average relevance of retrieved docs (0-1)
    if state.retrieved_docs:
        avg_relevance = np.mean([d["score"] for d in state.retrieved_docs])
    else:
        avg_relevance = 0.0
    
    # Factor 3: Evidence density -- at least 1 doc per needed topic
    topic_doc_count = {}
    for doc in state.retrieved_docs:
        if doc["topic"] in state.topics_needed:
            topic_doc_count[doc["topic"]] = topic_doc_count.get(doc["topic"], 0) + 1
    if state.topics_needed:
        topics_with_docs = sum(1 for t in state.topics_needed if topic_doc_count.get(t, 0) >= 1)
        evidence_density = topics_with_docs / len(state.topics_needed)
    else:
        evidence_density = 1.0
    
    # Combined score (weighted)
    state.sufficiency_score = (
        0.50 * topic_coverage +
        0.20 * avg_relevance +
        0.30 * evidence_density
    )
    state.is_sufficient = state.sufficiency_score >= SUFFICIENCY_THRESHOLD
    
    decision = "SUFFICIENT" if state.is_sufficient else "INSUFFICIENT"
    state.decision_log.append(
        f"EVALUATE (round {state.round_num}): "
        f"topic_cov={topic_coverage:.2f}, relevance={avg_relevance:.2f}, "
        f"evidence={evidence_density:.2f} -> score={state.sufficiency_score:.3f} "
        f"[{decision}] (threshold={SUFFICIENCY_THRESHOLD})"
    )
    return state


print("EVALUATE node defined.")


## Node 4: Reformulate Query

When context is insufficient, the agent reformulates its search query
to target **missing topic areas**. This is the key difference from
simple retry -- the agent learns from what it has and asks for what
it doesn't.

In production, an LLM generates the reformulated query. Here we
construct it from the missing topics and the original question.


In [ ]:
def node_reformulate(state: AgentState) -> AgentState:
    """REFORMULATE node: generate a new query targeting missing topics."""
    missing = state.topics_missing
    
    if missing:
        # Build a query targeting the missing topic areas
        topic_phrases = {
            "pricing": "TechStart pricing plans and costs",
            "database": "TechStart database support and performance",
            "security": "TechStart security certifications and encryption",
            "performance": "TechStart performance metrics and SLA",
            "deployment": "TechStart deployment infrastructure and disaster recovery",
            "api": "TechStart API SDK and integrations",
        }
        missing_phrases = [topic_phrases.get(t, f"TechStart {t}") for t in missing]
        state.search_query = " ".join(missing_phrases)
    else:
        # Topics covered but relevance too low -- rephrase original
        state.search_query = f"detailed information about {state.question}"
    
    state.decision_log.append(
        f"REFORMULATE: missing_topics={missing}, "
        f"new_query='{state.search_query}'"
    )
    return state


print("REFORMULATE node defined.")


## The Agent Loop

This is the core of agentic RAG. The `run_agent()` function
orchestrates the state machine:

```python
while round < max_rounds:
    state = retrieve(state)
    state = evaluate(state)
    if state.is_sufficient:
        break
    state = reformulate(state)
```

This maps directly to a LangGraph `StateGraph` with conditional edges.


In [ ]:
def run_agent(question: str, verbose: bool = True) -> AgentState:
    """Run the agentic RAG pipeline."""
    # Initialize state
    state = AgentState(question=question)
    state.topics_needed = detect_topics(question)
    state.search_query = question
    
    state.decision_log.append(
        f"START: question='{question}', topics_needed={state.topics_needed}"
    )
    
    if verbose:
        print(f"Question: {question}")
        print(f"Topics needed: {state.topics_needed}")
        print("-" * 60)
    
    while state.round_num < MAX_ROUNDS:
        # Node: RETRIEVE
        state = node_retrieve(state)
        if verbose:
            new_docs = state.retrieved_docs[-K:]  # last K docs
            print(f"Round {state.round_num}: Retrieved {[d['id'] for d in new_docs]} "
                  f"(topics: {set(d['topic'] for d in new_docs)})")
        
        # Node: EVALUATE
        state = node_evaluate(state)
        if verbose:
            status = "SUFFICIENT" if state.is_sufficient else "INSUFFICIENT"
            print(f"  Sufficiency: {state.sufficiency_score:.3f} [{status}]")
            print(f"  Topics covered: {state.topics_covered} / needed: {state.topics_needed}")
        
        if state.is_sufficient:
            if verbose:
                print(f"  -> ANSWER (enough context after {state.round_num} round(s))")
            break
        
        if state.round_num >= MAX_ROUNDS:
            if verbose:
                print(f"  -> ANSWER (max rounds reached)")
            break
        
        # Node: REFORMULATE
        state = node_reformulate(state)
        if verbose:
            print(f"  -> REFORMULATE: missing {state.topics_missing}")
            print(f"     New query: '{state.search_query}'")
    
    return state


print("Agent loop defined.")


## Demo 1: Simple Single-Topic Question

A simple question about pricing should be answered in 1 round.
The agent retrieves pricing docs and finds sufficient coverage.


In [ ]:
state1 = run_agent("What is the pricing for TechStart Pro plan?")
print(f"\nTotal docs retrieved: {len(state1.retrieved_docs)}")
print(f"Rounds used: {state1.round_num}")


## Demo 2: Cross-Topic Question

"Compare pricing and performance" needs docs from two topic areas.
Round 1 will likely retrieve mostly one topic. The agent should
detect the gap and retrieve more.


In [ ]:
state2 = run_agent("Compare the pricing and performance of TechStart")
print(f"\nTotal docs retrieved: {len(state2.retrieved_docs)}")
print(f"Rounds used: {state2.round_num}")
print(f"Topics covered: {state2.topics_covered}")


## Demo 3: Complex Multi-Topic Question

"What security certifications exist and how do they affect the
Enterprise deployment options?" -- needs security + deployment.


In [ ]:
state3 = run_agent("What security certifications does TechStart have and what deployment options exist for Enterprise?")
print(f"\nTotal docs retrieved: {len(state3.retrieved_docs)}")
print(f"Rounds used: {state3.round_num}")
print(f"Topics covered: {state3.topics_covered}")


## Demo 4: Broad Question Requiring Multiple Rounds

"Tell me everything about TechStart databases and their API access"
needs database + API docs. The agent must recognize both needs.


In [ ]:
state4 = run_agent("Tell me about TechStart database support and API access")
print(f"\nTotal docs retrieved: {len(state4.retrieved_docs)}")
print(f"Rounds used: {state4.round_num}")
print(f"Topics covered: {state4.topics_covered}")


## Decision Trace Analysis

Let's examine the full decision log for the cross-topic question.
This is what you would see in production monitoring.


In [ ]:
print("Decision trace for: 'Compare the pricing and performance of TechStart'")
print("=" * 70)
for entry in state2.decision_log:
    print(f"  {entry}")


## Single-Pass Retriever (Baseline)

For comparison, a standard RAG system that retrieves once with
no evaluation or follow-up.


In [ ]:
def single_pass_retrieve(question: str, k: int = K) -> List[Dict]:
    """Standard single-pass retrieval -- no agent loop."""
    return retrieve(question, k=k)


print("Single-pass baseline defined.")


## Evaluation Setup

We evaluate on queries classified by difficulty:
- **Single-topic**: Needs docs from 1 topic area
- **Cross-topic**: Needs docs from 2 topic areas
- **Multi-topic**: Needs docs from 3+ topic areas


In [ ]:
eval_queries = [
    # Single-topic (agent should match single-pass)
    {"query": "What is TechStart pricing?",
     "relevant": ["P01", "P02", "P03"], "type": "single"},
    {"query": "What databases does TechStart support?",
     "relevant": ["D01", "D02", "D03"], "type": "single"},
    {"query": "What security certifications does TechStart have?",
     "relevant": ["S01", "S02"], "type": "single"},
    {"query": "How does TechStart auto-scaling work?",
     "relevant": ["R01", "R02"], "type": "single"},

    # Cross-topic (agent should outperform)
    {"query": "Compare TechStart pricing plans and database support",
     "relevant": ["P01", "P02", "D01", "D03"], "type": "cross"},
    {"query": "What security features are included in the Enterprise deployment?",
     "relevant": ["S02", "S04", "T03", "T04"], "type": "cross"},
    {"query": "How do TechStart API rate limits vary by pricing tier?",
     "relevant": ["A01", "P01", "P02"], "type": "cross"},
    {"query": "What performance SLA comes with each pricing plan?",
     "relevant": ["R03", "P01", "P03"], "type": "cross"},

    # Multi-topic (agent should clearly outperform)
    {"query": "Give me a full overview of TechStart security, pricing, and deployment options",
     "relevant": ["S01", "S02", "P01", "P03", "T01", "T03"], "type": "multi"},
    {"query": "How do databases, caching, and API access work together in TechStart?",
     "relevant": ["D01", "D04", "A01", "A02"], "type": "multi"},
]

print(f"Evaluation set: {len(eval_queries)} queries")
for qtype in ["single", "cross", "multi"]:
    count = sum(1 for q in eval_queries if q["type"] == qtype)
    print(f"  {qtype}: {count} queries")


## Evaluation Metrics

In [ ]:
def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    top_k = retrieved_ids[:k]
    hits = sum(1 for r in relevant_ids if r in top_k)
    return hits / len(relevant_ids) if relevant_ids else 0.0


def mrr_score(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


def topic_recall(retrieved_docs: List[Dict], relevant_ids: List[str]) -> float:
    """What fraction of needed topic areas have at least one retrieved doc?"""
    needed_topics = set()
    for doc in corpus:
        if doc["id"] in relevant_ids:
            needed_topics.add(doc["topic"])
    if not needed_topics:
        return 1.0
    covered = set(d["topic"] for d in retrieved_docs if d["id"] in relevant_ids)
    return len(covered) / len(needed_topics)


print("Metrics defined: recall@k, MRR, topic_recall")


## Benchmark: Single-Pass vs Agentic Retrieval

The key comparison. We use the same K for single-pass.
Note: the agent retrieves K docs per round, so with 2 rounds it
gets 2K docs. For fair comparison, we also test single-pass with 2K.


In [ ]:
benchmark = {"single_pass_k3": [], "single_pass_k6": [], "agentic": []}

for q in eval_queries:
    query_text = q["query"]
    relevant = q["relevant"]
    
    # Single-pass K=3
    sp3 = single_pass_retrieve(query_text, k=3)
    sp3_ids = [d["id"] for d in sp3]
    benchmark["single_pass_k3"].append({
        "query": query_text, "type": q["type"],
        "recall": recall_at_k(sp3_ids, relevant, len(sp3_ids)),
        "mrr": mrr_score(sp3_ids, relevant),
        "topic_recall": topic_recall(sp3, relevant),
        "docs": sp3_ids, "rounds": 1,
    })
    
    # Single-pass K=6 (same budget as 2 agent rounds)
    sp6 = single_pass_retrieve(query_text, k=6)
    sp6_ids = [d["id"] for d in sp6]
    benchmark["single_pass_k6"].append({
        "query": query_text, "type": q["type"],
        "recall": recall_at_k(sp6_ids, relevant, len(sp6_ids)),
        "mrr": mrr_score(sp6_ids, relevant),
        "topic_recall": topic_recall(sp6, relevant),
        "docs": sp6_ids, "rounds": 1,
    })
    
    # Agentic retrieval
    agent_state = run_agent(query_text, verbose=False)
    agent_ids = agent_state.retrieved_ids
    benchmark["agentic"].append({
        "query": query_text, "type": q["type"],
        "recall": recall_at_k(agent_ids, relevant, len(agent_ids)),
        "mrr": mrr_score(agent_ids, relevant),
        "topic_recall": topic_recall(agent_state.retrieved_docs, relevant),
        "docs": agent_ids, "rounds": agent_state.round_num,
    })

print(f"Benchmark complete: {len(eval_queries)} queries x 3 methods")


## Aggregate Results

In [ ]:
print(f"{'Method':<20s} {'Recall':>8s} {'MRR':>8s} {'TopicR':>8s} {'AvgRnds':>8s}")
print("-" * 54)

for method_name, method_key in [("Single-pass (K=3)", "single_pass_k3"),
                                 ("Single-pass (K=6)", "single_pass_k6"),
                                 ("Agentic", "agentic")]:
    data = benchmark[method_key]
    avg_recall = np.mean([r["recall"] for r in data])
    avg_mrr = np.mean([r["mrr"] for r in data])
    avg_topic = np.mean([r["topic_recall"] for r in data])
    avg_rounds = np.mean([r["rounds"] for r in data])
    print(f"{method_name:<20s} {avg_recall:>8.3f} {avg_mrr:>8.3f} {avg_topic:>8.3f} {avg_rounds:>8.1f}")

# Improvement
print()
sp3_recall = np.mean([r["recall"] for r in benchmark["single_pass_k3"]])
agent_recall = np.mean([r["recall"] for r in benchmark["agentic"]])
delta = agent_recall - sp3_recall
pct = (delta / sp3_recall * 100) if sp3_recall > 0 else 0
print(f"Agentic vs Single-pass(K=3): {delta:+.3f} ({pct:+.1f}% recall)")

sp6_recall = np.mean([r["recall"] for r in benchmark["single_pass_k6"]])
delta6 = agent_recall - sp6_recall
pct6 = (delta6 / sp6_recall * 100) if sp6_recall > 0 else 0
print(f"Agentic vs Single-pass(K=6): {delta6:+.3f} ({pct6:+.1f}% recall)")


## Per-Query-Type Breakdown

The agentic approach should shine on cross-topic and multi-topic
queries where targeted follow-up retrieval helps.


In [ ]:
for qtype in ["single", "cross", "multi"]:
    print(f"\nQuery type: {qtype}")
    print(f"  {'Method':<20s} {'Recall':>8s} {'TopicR':>8s} {'Rounds':>8s}")
    print("  " + "-" * 38)
    for method_name, method_key in [("Single-pass (K=3)", "single_pass_k3"),
                                     ("Single-pass (K=6)", "single_pass_k6"),
                                     ("Agentic", "agentic")]:
        data = [r for r in benchmark[method_key] if r["type"] == qtype]
        if not data:
            continue
        avg_r = np.mean([r["recall"] for r in data])
        avg_t = np.mean([r["topic_recall"] for r in data])
        avg_rnds = np.mean([r["rounds"] for r in data])
        print(f"  {method_name:<20s} {avg_r:>8.3f} {avg_t:>8.3f} {avg_rnds:>8.1f}")


## Per-Query Detail

In [ ]:
print("Per-Query Comparison:\n")
for i, q in enumerate(eval_queries):
    sp3_r = benchmark["single_pass_k3"][i]["recall"]
    sp6_r = benchmark["single_pass_k6"][i]["recall"]
    ag_r = benchmark["agentic"][i]["recall"]
    ag_rounds = benchmark["agentic"][i]["rounds"]
    
    best = max(sp3_r, sp6_r, ag_r)
    winner = "AGENT WINS" if ag_r > sp3_r else ("TIE" if ag_r == sp3_r else "SP WINS")
    
    print(f"  Q{i+1:2d} [{q['type']:>6s}] SP3={sp3_r:.2f} SP6={sp6_r:.2f} "
          f"Agent={ag_r:.2f} (rounds={ag_rounds}) [{winner}]")


## Qualitative Analysis: Why Agentic Retrieval Helps

Let's examine specific cases where the agent outperformed single-pass.


In [ ]:
print("Cases Where Agentic Retrieval Outperformed Single-Pass\n")
print("=" * 70)

for i, q in enumerate(eval_queries):
    sp3_r = benchmark["single_pass_k3"][i]["recall"]
    ag_r = benchmark["agentic"][i]["recall"]
    
    if ag_r > sp3_r:
        print(f"\nQ{i+1}: {q['query']}")
        print(f"  Type: {q['type']} | Relevant: {q['relevant']}")
        print(f"  Single-pass (K=3): {benchmark['single_pass_k3'][i]['docs']} (recall={sp3_r:.2f})")
        print(f"  Agentic:           {benchmark['agentic'][i]['docs']} (recall={ag_r:.2f})")
        print(f"  Rounds used: {benchmark['agentic'][i]['rounds']}")
        
        # Show what topics single-pass missed
        sp_topics = set()
        ag_topics = set()
        for doc_id in benchmark["single_pass_k3"][i]["docs"]:
            for doc in corpus:
                if doc["id"] == doc_id:
                    sp_topics.add(doc["topic"])
        for doc_id in benchmark["agentic"][i]["docs"]:
            for doc in corpus:
                if doc["id"] == doc_id:
                    ag_topics.add(doc["topic"])
        print(f"  Single-pass topics: {sp_topics}")
        print(f"  Agentic topics: {ag_topics}")

print("\n" + "=" * 70)


## Error Analysis

Where did the agent fail or not help?


In [ ]:
print("Error Analysis\n")

# Cases where agent didn't improve
no_improvement = []
agent_worse = []
for i, q in enumerate(eval_queries):
    sp3_r = benchmark["single_pass_k3"][i]["recall"]
    ag_r = benchmark["agentic"][i]["recall"]
    if ag_r == sp3_r:
        no_improvement.append(i)
    elif ag_r < sp3_r:
        agent_worse.append(i)

if no_improvement:
    print(f"No improvement ({len(no_improvement)} queries):")
    for i in no_improvement:
        print(f"  Q{i+1} [{eval_queries[i]['type']}]: {eval_queries[i]['query'][:60]}")
        print(f"    Both achieved recall={benchmark['single_pass_k3'][i]['recall']:.2f}")
        if eval_queries[i]["type"] == "single":
            print(f"    (Expected: single-topic questions don't need follow-up)")

if agent_worse:
    print(f"\nAgent performed worse ({len(agent_worse)} queries):")
    for i in agent_worse:
        print(f"  Q{i+1} [{eval_queries[i]['type']}]: recall dropped from "
              f"{benchmark['single_pass_k3'][i]['recall']:.2f} to "
              f"{benchmark['agentic'][i]['recall']:.2f}")
else:
    print(f"\nAgent never performed worse than single-pass.")

# Round distribution
print(f"\nRound distribution:")
round_counts = {}
for r in benchmark["agentic"]:
    rnd = r["rounds"]
    round_counts[rnd] = round_counts.get(rnd, 0) + 1
for rnd in sorted(round_counts):
    print(f"  {rnd} round(s): {round_counts[rnd]} queries")


## Mapping to LangGraph

Our from-scratch implementation maps directly to LangGraph:

```python
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question: str
    search_query: str
    retrieved_docs: list
    sufficiency_score: float
    is_sufficient: bool

# Build graph
graph = StateGraph(AgentState)
graph.add_node('retrieve', node_retrieve)
graph.add_node('evaluate', node_evaluate)
graph.add_node('reformulate', node_reformulate)

# Edges
graph.set_entry_point('retrieve')
graph.add_edge('retrieve', 'evaluate')
graph.add_conditional_edges(
    'evaluate',
    lambda state: 'end' if state['is_sufficient'] else 'reformulate',
    {'end': END, 'reformulate': 'reformulate'}
)
graph.add_edge('reformulate', 'retrieve')

agent = graph.compile()
result = agent.invoke({'question': 'Compare pricing and performance'})
```

The concept is identical. LangGraph adds:
- **Persistence**: Save/resume agent state across calls
- **Streaming**: Stream node outputs for real-time UX
- **Human-in-loop**: Pause at decision nodes for human review
- **Checkpointing**: Save state at each node for debugging


## Limitations

1. **Rule-based topic detection.** Production systems use an LLM to
   analyze question intent. Our keyword matching misses nuanced topics.

2. **Heuristic sufficiency scoring.** An LLM evaluator would assess
   whether the actual text content answers the question, not just topic coverage.

3. **No generation step.** We evaluate retrieval quality only.
   In production, the agent generates an answer and may realize
   mid-generation that context is insufficient.

4. **Small corpus.** With 30 docs, even random retrieval has decent coverage.
   The agentic advantage grows with corpus size.

5. **Fixed reformulation.** Our reformulation is template-based.
   LLM-powered reformulation adapts to what was actually retrieved.

6. **No cost-awareness.** Each retrieval round costs latency and compute.
   Production agents balance quality vs cost.


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| No max rounds limit | Agent loops forever on impossible questions | Always cap rounds (2-3) |
| Same query each round | Retrieving same docs repeatedly | Exclude already-retrieved docs |
| No sufficiency threshold | Agent always does max rounds | Evaluate and stop early when sufficient |
| Over-engineering decisions | Complex state machines for simple questions | Simple questions should use 1 round |
| No monitoring | Can't debug why agent made bad decisions | Log every decision with reasoning |
| Ignoring latency | Each round adds 100-300ms | Optimize for 1-round on 80% of queries |


## Mini Challenge

1. **LLM evaluator.** Replace the heuristic sufficiency scorer with an
   LLM that reads retrieved docs and decides "can I answer this question?"

2. **Add generation.** After retrieval, generate an answer using the
   retrieved context. Compare answer quality single-pass vs agentic.

3. **LangGraph implementation.** Port this notebook to actual LangGraph.
   Add streaming, checkpointing, and human-in-loop breakpoints.

4. **Adaptive K.** Instead of fixed K=3 per round, let the agent choose
   how many docs to retrieve based on question complexity.

5. **Tool-use agent.** Give the agent multiple tools: web search,
   SQL query, vector search. Let it choose which tool to use per round.


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Latency budget** | Set per-round timeout (200ms). Total budget = rounds x timeout |
| **Cost control** | Cache sufficiency decisions. Most queries need 1 round |
| **Observability** | Log full decision trace per query for debugging |
| **Fallback** | If agent errors, fall back to single-pass retrieval |
| **A/B testing** | Compare single-pass vs agentic on production traffic |
| **Query routing** | Classify simple vs complex upfront. Route simple to single-pass |
| **LangGraph persistence** | Save agent state for conversation continuity |
| **Human-in-loop** | On low-confidence queries, pause for human review |


## How to Improve This Project

1. **Use LangGraph** for production-grade state management, streaming,
   and persistence.
2. **Add an LLM** for sufficiency evaluation, query reformulation,
   and answer generation.
3. **Combine with hybrid search** (BM25 + dense) from project 21.
4. **Add query rewriting** from project 22 as a pre-processing step.
5. **Implement cross-encoder reranking** after each retrieval round.
6. **Test on a larger corpus** (1000+ documents) where agentic
   retrieval provides more value.
7. **Add tool use** -- let the agent choose between vector search,
   keyword search, SQL queries, and web search.


## Key Takeaways

1. **Standard RAG retrieves once and hopes.** Agentic RAG evaluates
   what it got and retrieves more if needed.

2. **The agent loop is simple:** Retrieve -> Evaluate -> Reformulate -> Retrieve.
   The power is in the decision logic, not the complexity.

3. **Cross-topic questions benefit most.** Single-topic questions work
   fine with single-pass. Multi-topic questions need targeted follow-up.

4. **Sufficiency scoring is the key decision.** If you get this wrong,
   the agent either wastes rounds or stops too early.

5. **Exclude already-retrieved docs.** Without this, follow-up rounds
   just return the same documents.

6. **Decision logging is essential.** Without trace logs, you cannot
   debug why the agent made a particular decision.

7. **This maps directly to LangGraph.** The state machine pattern
   (StateGraph + conditional edges) is the foundation of all LangGraph agents.

8. **Balance quality vs latency.** Most queries should resolve in 1 round.
   Only complex questions should trigger follow-up retrieval.
